## Random Forest Regression

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [4]:
df = pd.read_csv('cardekho_imputated.csv', index_col=[0])

In [5]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [7]:
# drop the unnecessary columns

df.drop(columns=['car_name', 'brand'], axis=1, inplace=True)

In [8]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [9]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15411 entries, 0 to 19543
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   model              15411 non-null  object 
 1   vehicle_age        15411 non-null  int64  
 2   km_driven          15411 non-null  int64  
 3   seller_type        15411 non-null  object 
 4   fuel_type          15411 non-null  object 
 5   transmission_type  15411 non-null  object 
 6   mileage            15411 non-null  float64
 7   engine             15411 non-null  int64  
 8   max_power          15411 non-null  float64
 9   seats              15411 non-null  int64  
 10  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(5), object(4)
memory usage: 1.4+ MB


In [11]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']

In [12]:
# independant and dependant features

x = df.drop(columns=['selling_price'], axis=1)
y = df['selling_price']

In [13]:
x.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [15]:
print(df['fuel_type'].value_counts(), df['fuel_type'].unique())
print(df['seller_type'].value_counts(), df['seller_type'].unique())
print(df['fuel_type'].value_counts(), df['fuel_type'].unique())
print(df['transmission_type'].value_counts(), df['transmission_type'].unique())

fuel_type
Petrol      7643
Diesel      7419
CNG          301
LPG           44
Electric       4
Name: count, dtype: int64 ['Petrol' 'Diesel' 'CNG' 'LPG' 'Electric']
seller_type
Dealer              9539
Individual          5699
Trustmark Dealer     173
Name: count, dtype: int64 ['Individual' 'Dealer' 'Trustmark Dealer']
fuel_type
Petrol      7643
Diesel      7419
CNG          301
LPG           44
Electric       4
Name: count, dtype: int64 ['Petrol' 'Diesel' 'CNG' 'LPG' 'Electric']
transmission_type
Manual       12225
Automatic     3186
Name: count, dtype: int64 ['Manual' 'Automatic']


In [23]:
# feature encoding and scaling

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [17]:
le = LabelEncoder()
x['model'] = le.fit_transform(x['model'])

In [19]:
pd.DataFrame(x['model'])

,model
0,7
1,54
2,118
3,7
4,38
...,...
19537,117
19540,42
19541,77
19542,114


In [20]:
num_features

['vehicle_age',
 'km_driven',
 'mileage',
 'engine',
 'max_power',
 'seats',
 'selling_price']

In [21]:
one_hot_columns = ['transmission_type', 'seller_type', 'fuel_type']
num_features = x.select_dtypes(exclude='object').columns

In [24]:
one_hot = OneHotEncoder()
scaler = StandardScaler()

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ('OneHotEncoder',one_hot, one_hot_columns),
        ('StandardScaler', scaler, num_features)
    ], remainder='passthrough'
)

In [27]:
x = preprocessor.fit_transform(x)

In [28]:
# trian test split

from sklearn.model_selection import train_test_split

In [30]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## Model Training

In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [36]:
# create a function to evaluate the model

def model_score(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    r2score = r2_score(true, predicted)

    return mae, mse, r2score

In [37]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'K-Neighbors': KNeighborsRegressor()
}

In [38]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train, y_train)

    # make predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # evaluate the model
    model_train_mae, model_train_mse, model_train_r2_score = model_score(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_r2_score = model_score(y_test, y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance on Training Set')
    print("- MAE: {:.4f}".format(model_train_mae))
    print('- MSE: {:.4f}'.format(model_train_mse))
    print('- R2Score: {:.4f}'.format(model_train_r2_score))
    print('-'*30)
    print('Model Performance on Test Set')
    print("- MAE: {:.4f}".format(model_test_mae))
    print('- MSE: {:.4f}'.format(model_test_mse))
    print('- R2Score: {:.4f}'.format(model_test_r2_score))
    print('\n')

Linear Regression
Model Performance on Training Set
- MAE: 268101.6071
- MSE: 306756099359.7596
- R2Score: 0.6218
------------------------------
Model Performance on Test Set
- MAE: 279618.5794
- MSE: 252550062888.5655
- R2Score: 0.6645


Decision Tree Regressor
Model Performance on Training Set
- MAE: 5164.8199
- MSE: 432524990.5364
- R2Score: 0.9995
------------------------------
Model Performance on Test Set
- MAE: 126113.4906
- MSE: 95132496548.0277
- R2Score: 0.8736


Random Forest Regressor
Model Performance on Training Set
- MAE: 40194.6321
- MSE: 20129607197.8626
- R2Score: 0.9752
------------------------------
Model Performance on Test Set
- MAE: 102597.7055
- MSE: 53706026641.6762
- R2Score: 0.9287


Lasso
Model Performance on Training Set
- MAE: 268099.3635
- MSE: 306756104063.1791
- R2Score: 0.6218
------------------------------
Model Performance on Test Set
- MAE: 279614.9111
- MSE: 252549104707.3054
- R2Score: 0.6645


Ridge
Model Performance on Training Set
- MAE: 268060

In [39]:
# hyperparameter tuning the model

knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

In [40]:
# models list

randomcv_models = [
    ('KNeighbors', KNeighborsRegressor(), knn_params),
    ('RandomForest', RandomForestRegressor(), rf_params)
]

In [41]:
from sklearn.model_selection import RandomizedSearchCV

In [42]:
model_param = {}

for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model, param_distributions=params, n_iter=100, cv=3, verbose=2, n_jobs=-1)
    random.fit(x_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f'Best parameters for {model_name}:\n')
    print(model_param[model_name])

d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=100. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits


d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
84 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
21 fits failed with the following error:
Traceback (most recent call last):
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\s

Best parameters for KNeighbors:

{'n_neighbors': 10}
Best parameters for RandomForest:

{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}


In [43]:
models ={
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_split=2, max_features=8, max_depth=15, n_jobs=-1),
    "KNeighbors": KNeighborsRegressor(n_neighbors=10, n_jobs=-1)
}

In [44]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train, y_train)

    # make predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # evaluate the model
    model_train_mae, model_train_mse, model_train_r2_score = model_score(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_r2_score = model_score(y_test, y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance on Training Set')
    print("- MAE: {:.4f}".format(model_train_mae))
    print('- MSE: {:.4f}'.format(model_train_mse))
    print('- R2Score: {:.4f}'.format(model_train_r2_score))
    print('-'*30)
    print('Model Performance on Test Set')
    print("- MAE: {:.4f}".format(model_test_mae))
    print('- MSE: {:.4f}'.format(model_test_mse))
    print('- R2Score: {:.4f}'.format(model_test_r2_score))
    print('\n')

Random Forest
Model Performance on Training Set
- MAE: 54361.3426
- MSE: 17460927085.2997
- R2Score: 0.9785
------------------------------
Model Performance on Test Set
- MAE: 98140.2681
- MSE: 46882793504.9883
- R2Score: 0.9377


KNeighbors
Model Performance on Training Set
- MAE: 104817.0750
- MSE: 133895286944.3543
- R2Score: 0.8349
------------------------------
Model Performance on Test Set
- MAE: 119078.5031
- MSE: 71473425486.7418
- R2Score: 0.9051


